# Classifying Penguins with Keras

In [29]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn import preprocessing
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix, precision_score, recall_score, f1_score, roc_auc_score, roc_curve

In [30]:
! pip install palmerpenguins
from palmerpenguins import load_penguins
penguins = load_penguins()
penguins = penguins.dropna()
penguins.head()


,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex,year
0,Adelie,Torgersen,39.1,18.7,181.0,3750.0,male,2007
1,Adelie,Torgersen,39.5,17.4,186.0,3800.0,female,2007
2,Adelie,Torgersen,40.3,18.0,195.0,3250.0,female,2007
4,Adelie,Torgersen,36.7,19.3,193.0,3450.0,female,2007
5,Adelie,Torgersen,39.3,20.6,190.0,3650.0,male,2007


In [31]:
#SHUFFLE THE DATA
penguins = penguins.sample(frac=1).reset_index(drop=True)
penguins.head()


,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex,year
0,Gentoo,Biscoe,50.1,15.0,225.0,5000.0,male,2008
1,Gentoo,Biscoe,49.9,16.1,213.0,5400.0,male,2009
2,Adelie,Dream,43.2,18.5,192.0,4100.0,male,2008
3,Adelie,Biscoe,41.6,18.0,192.0,3950.0,male,2008
4,Adelie,Biscoe,37.6,19.1,194.0,3750.0,male,2008


In [32]:
penguins_x = pd.concat([penguins[['body_mass_g', 'bill_length_mm', 'bill_depth_mm', 'flipper_length_mm']], pd.get_dummies(penguins['sex'])], axis = 1)
# penguins_x = penguins_x[['body_mass_g', 'bill_length_mm', 'bill_depth_mm', 'flipper_length_mm', 'female', 'male']]
penguins_x

,body_mass_g,bill_length_mm,bill_depth_mm,flipper_length_mm,female,male
0,5000.0,50.1,15.0,225.0,False,True
1,5400.0,49.9,16.1,213.0,False,True
2,4100.0,43.2,18.5,192.0,False,True
3,3950.0,41.6,18.0,192.0,False,True
4,3750.0,37.6,19.1,194.0,False,True
...,...,...,...,...,...,...
328,4000.0,42.1,19.1,195.0,False,True
329,4150.0,52.0,19.0,197.0,False,True
330,4400.0,45.1,14.4,210.0,True,False
331,4775.0,43.2,19.0,197.0,False,True


In [33]:
x = penguins_x.values
min_max_scaler = preprocessing.MinMaxScaler()
scaled_penguins_x = pd.DataFrame(min_max_scaler.fit_transform(x), columns=penguins_x.columns)
scaled_penguins_x

,body_mass_g,bill_length_mm,bill_depth_mm,flipper_length_mm,female,male
0,0.638889,0.654545,0.226190,0.898305,0.0,1.0
1,0.750000,0.647273,0.357143,0.694915,0.0,1.0
2,0.388889,0.403636,0.642857,0.338983,0.0,1.0
3,0.347222,0.345455,0.583333,0.338983,0.0,1.0
4,0.291667,0.200000,0.714286,0.372881,0.0,1.0
...,...,...,...,...,...,...
328,0.361111,0.363636,0.714286,0.389831,0.0,1.0
329,0.402778,0.723636,0.702381,0.423729,0.0,1.0
330,0.472222,0.472727,0.154762,0.644068,1.0,0.0
331,0.576389,0.403636,0.702381,0.423729,0.0,1.0


In [34]:
penguins_y = penguins['species']
print(penguins_y)
penguins_y = penguins_y.astype('category').cat.codes.to_numpy()
penguins_y

0         Gentoo
1         Gentoo
2         Adelie
3         Adelie
4         Adelie
         ...    
328       Adelie
329    Chinstrap
330       Gentoo
331       Adelie
332       Gentoo
Name: species, Length: 333, dtype: str


array([2, 2, 0, 0, 0, 0, 2, 0, 0, 1, 1, 0, 2, 1, 1, 0, 2, 0, 2, 0, 2, 0,
       1, 2, 0, 1, 0, 1, 1, 0, 0, 1, 0, 2, 2, 0, 2, 0, 2, 2, 1, 2, 1, 0,
       1, 0, 0, 0, 2, 2, 1, 2, 2, 0, 1, 1, 1, 0, 0, 2, 0, 0, 2, 0, 0, 2,
       2, 2, 0, 0, 0, 2, 2, 0, 0, 2, 0, 2, 0, 2, 1, 0, 0, 0, 2, 0, 1, 1,
       0, 1, 2, 0, 1, 0, 0, 1, 0, 0, 2, 0, 2, 0, 1, 2, 1, 2, 0, 0, 0, 2,
       2, 0, 0, 2, 1, 0, 2, 1, 1, 0, 0, 0, 2, 2, 2, 0, 2, 1, 2, 0, 2, 0,
       0, 2, 2, 1, 1, 2, 0, 0, 1, 1, 0, 0, 0, 0, 0, 1, 1, 2, 0, 2, 1, 2,
       0, 2, 2, 2, 0, 2, 2, 0, 0, 0, 0, 2, 2, 0, 0, 2, 0, 0, 0, 0, 1, 1,
       2, 2, 0, 1, 0, 0, 0, 0, 2, 2, 0, 2, 2, 2, 0, 2, 2, 1, 0, 2, 2, 0,
       1, 0, 0, 2, 1, 0, 0, 0, 0, 1, 0, 1, 2, 0, 1, 2, 2, 0, 2, 1, 1, 1,
       0, 2, 2, 0, 0, 0, 0, 0, 1, 2, 0, 0, 0, 0, 2, 0, 2, 0, 2, 2, 2, 0,
       2, 0, 0, 2, 1, 0, 0, 0, 2, 1, 0, 1, 1, 2, 2, 2, 0, 2, 2, 1, 1, 1,
       2, 1, 0, 2, 2, 1, 1, 2, 0, 0, 2, 1, 1, 2, 0, 1, 0, 2, 0, 2, 1, 0,
       0, 0, 2, 2, 2, 1, 0, 2, 2, 0, 1, 0, 0, 0, 0,

In [35]:
#construct the model
inputs = keras.Input(shape=(6,))
x = layers.Dense(7, activation = 'relu')(inputs)
x = layers.Dense(5, activation = 'relu')(x)
x = layers.Dense(3, activation = 'relu')(x)
outputs = layers.Dense(3, activation='softmax')(x)
model = keras.Model(inputs=inputs, outputs=outputs, name="penguin_model")

In [36]:
model.summary()

Model: "penguin_model"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_2 (InputLayer)      │ (None, 6)              │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ (None, 7)              │            49 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_9 (Dense)                 │ (None, 5)              │            40 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_10 (Dense)                │ (None, 3)              │            18 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_11 (Dense)                │ (None, 3)              │            12 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 119 (476.00 B)

 Trainable params: 119 (476.00 B)

 Non-trainable params: 0 (0.00 B)

In [37]:
keras.utils.plot_model(model, show_shapes = True)

You must install pydot (`pip install pydot`) for `plot_model` to work.


In [38]:
model.compile(
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    optimizer=keras.optimizers.RMSprop(),
    metrics=["accuracy"],
)

history = model.fit(scaled_penguins_x, penguins_y, batch_size = 64, epochs=100, validation_split=0.1)

scores = model.evaluate(scaled_penguins_x, penguins_y, verbose=2)

Epoch 1/100


/Users/mkenne16/Documents/Advanced Machine Learning/week 7/.conda-env/lib/python3.12/site-packages/keras/src/backend/tensorflow/nn.py:1216: UserWarning: "`sparse_categorical_crossentropy` received `from_logits=True`, but the `output` argument was produced by a Softmax activation and thus does not represent logits. Was this intended?
  output, from_logits = _get_logits(


5/5 ━━━━━━━━━━━━━━━━━━━━ 2s 128ms/step - accuracy: 0.4047 - loss: 1.0973 - val_accuracy: 0.4118 - val_loss: 1.0946
Epoch 2/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.4415 - loss: 1.0955 - val_accuracy: 0.4118 - val_loss: 1.0928
Epoch 3/100
1/5 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - accuracy: 0.3750 - loss: 1.0948

/Users/mkenne16/Documents/Advanced Machine Learning/week 7/.conda-env/lib/python3.12/site-packages/keras/src/backend/tensorflow/nn.py:1216: UserWarning: "`sparse_categorical_crossentropy` received `from_logits=True`, but the `output` argument was produced by a Softmax activation and thus does not represent logits. Was this intended?
  output, from_logits = _get_logits(


5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.4415 - loss: 1.0944 - val_accuracy: 0.4118 - val_loss: 1.0912
Epoch 4/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - accuracy: 0.4415 - loss: 1.0934 - val_accuracy: 0.4118 - val_loss: 1.0898
Epoch 5/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 0.4415 - loss: 1.0924 - val_accuracy: 0.4118 - val_loss: 1.0885
Epoch 6/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.4415 - loss: 1.0915 - val_accuracy: 0.4118 - val_loss: 1.0874
Epoch 7/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.4415 - loss: 1.0907 - val_accuracy: 0.4118 - val_loss: 1.0862
Epoch 8/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.4415 - loss: 1.0898 - val_accuracy: 0.4118 - val_loss: 1.0850
Epoch 9/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.4415 - loss: 1.0890 - val_accuracy: 0.4118 - val_loss: 1.0838
Epoch 10/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.4415 - loss: 1.0882 - val_accuracy: 0.4118 - val_loss: 1.0827
Epo

In [39]:
model_logit_true = keras.Model(inputs=inputs, outputs=outputs, name="penguin_model_scaled")

model_logit_true.compile(
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=False),
    optimizer=keras.optimizers.RMSprop(),
    metrics=["accuracy"],
)

history_logit_true = model_logit_true.fit(scaled_penguins_x, penguins_y, batch_size = 64, epochs = 100, validation_split = 0.1)

scores = model_logit_true.evaluate(scaled_penguins_x, penguins_y, verbose = 2)

Epoch 1/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 1s 39ms/step - accuracy: 0.4415 - loss: 1.0588 - val_accuracy: 0.4118 - val_loss: 1.0266
Epoch 2/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.4415 - loss: 1.0588 - val_accuracy: 0.4118 - val_loss: 1.0263
Epoch 3/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.4415 - loss: 1.0587 - val_accuracy: 0.4118 - val_loss: 1.0261
Epoch 4/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.4415 - loss: 1.0587 - val_accuracy: 0.4118 - val_loss: 1.0260
Epoch 5/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step - accuracy: 0.4415 - loss: 1.0586 - val_accuracy: 0.4118 - val_loss: 1.0258
Epoch 6/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.4415 - loss: 1.0586 - val_accuracy: 0.4118 - val_loss: 1.0256
Epoch 7/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.4415 - loss: 1.0586 - val_accuracy: 0.4118 - val_loss: 1.0254
Epoch 8/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.4415 - loss: 1.0585 - val_accuracy: 0.4118 - val_loss:

In [40]:
model_logit_true.predict(scaled_penguins_x)

11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step 


array([[0.4406446 , 0.2153228 , 0.34403268],
       [0.4406446 , 0.2153228 , 0.34403268],
       [0.4406446 , 0.2153228 , 0.34403268],
       [0.4406446 , 0.2153228 , 0.34403268],
       [0.4406446 , 0.2153228 , 0.34403268],
       [0.4406446 , 0.2153228 , 0.34403268],
       [0.4406446 , 0.2153228 , 0.34403268],
       [0.4406446 , 0.2153228 , 0.34403268],
       [0.4406446 , 0.2153228 , 0.34403268],
       [0.4406446 , 0.2153228 , 0.34403268],
       [0.4406446 , 0.2153228 , 0.34403268],
       [0.4406446 , 0.2153228 , 0.34403268],
       [0.4406446 , 0.2153228 , 0.34403268],
       [0.4406446 , 0.2153228 , 0.34403268],
       [0.4406446 , 0.2153228 , 0.34403268],
       [0.4406446 , 0.2153228 , 0.34403268],
       [0.4406446 , 0.2153228 , 0.34403268],
       [0.4406446 , 0.2153228 , 0.34403268],
       [0.4406446 , 0.2153228 , 0.34403268],
       [0.4406446 , 0.2153228 , 0.34403268],
       [0.4406446 , 0.2153228 , 0.34403268],
       [0.4406446 , 0.2153228 , 0.34403268],
       [0.

In [41]:
penguins['species']

0         Gentoo
1         Gentoo
2         Adelie
3         Adelie
4         Adelie
         ...    
328       Adelie
329    Chinstrap
330       Gentoo
331       Adelie
332       Gentoo
Name: species, Length: 333, dtype: str

In [42]:
penguins_y

array([2, 2, 0, 0, 0, 0, 2, 0, 0, 1, 1, 0, 2, 1, 1, 0, 2, 0, 2, 0, 2, 0,
       1, 2, 0, 1, 0, 1, 1, 0, 0, 1, 0, 2, 2, 0, 2, 0, 2, 2, 1, 2, 1, 0,
       1, 0, 0, 0, 2, 2, 1, 2, 2, 0, 1, 1, 1, 0, 0, 2, 0, 0, 2, 0, 0, 2,
       2, 2, 0, 0, 0, 2, 2, 0, 0, 2, 0, 2, 0, 2, 1, 0, 0, 0, 2, 0, 1, 1,
       0, 1, 2, 0, 1, 0, 0, 1, 0, 0, 2, 0, 2, 0, 1, 2, 1, 2, 0, 0, 0, 2,
       2, 0, 0, 2, 1, 0, 2, 1, 1, 0, 0, 0, 2, 2, 2, 0, 2, 1, 2, 0, 2, 0,
       0, 2, 2, 1, 1, 2, 0, 0, 1, 1, 0, 0, 0, 0, 0, 1, 1, 2, 0, 2, 1, 2,
       0, 2, 2, 2, 0, 2, 2, 0, 0, 0, 0, 2, 2, 0, 0, 2, 0, 0, 0, 0, 1, 1,
       2, 2, 0, 1, 0, 0, 0, 0, 2, 2, 0, 2, 2, 2, 0, 2, 2, 1, 0, 2, 2, 0,
       1, 0, 0, 2, 1, 0, 0, 0, 0, 1, 0, 1, 2, 0, 1, 2, 2, 0, 2, 1, 1, 1,
       0, 2, 2, 0, 0, 0, 0, 0, 1, 2, 0, 0, 0, 0, 2, 0, 2, 0, 2, 2, 2, 0,
       2, 0, 0, 2, 1, 0, 0, 0, 2, 1, 0, 1, 1, 2, 2, 2, 0, 2, 2, 1, 1, 1,
       2, 1, 0, 2, 2, 1, 1, 2, 0, 0, 2, 1, 1, 2, 0, 1, 0, 2, 0, 2, 1, 0,
       0, 0, 2, 2, 2, 1, 0, 2, 2, 0, 1, 0, 0, 0, 0,